# Notebook 2: XGBoost Forecasting Baseline
Fastest to train, strong benchmark. Uses engineered lag + time features.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from preprocessing import preprocess
from evaluate import report
from visualize import plot_forecast
import joblib

plt.rcParams['figure.figsize'] = (16, 5)

In [ ]:
df = preprocess('../data/energy_consumption.csv')
print(df.shape)
df.head()

In [ ]:
# Train/val/test split — last 30 days = test, prior 30 days = val
TARGET = 'consumption_mwh'
FEATURES = [c for c in df.columns if c != TARGET]

test_cutoff = df.index[-24*30]
val_cutoff  = df.index[-24*60]

train = df[df.index < val_cutoff]
val   = df[(df.index >= val_cutoff) & (df.index < test_cutoff)]
test  = df[df.index >= test_cutoff]

print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    random_state=42,
    tree_method='hist'
)

model.fit(
    train[FEATURES], train[TARGET],
    eval_set=[(val[FEATURES], val[TARGET])],
    verbose=100
)

joblib.dump(model, '../models/xgboost_model.pkl')
print('Model saved.')

In [ ]:
preds = model.predict(test[FEATURES])
metrics = report(test[TARGET], preds, 'XGBoost')
plot_forecast(test[TARGET], preds, title='XGBoost — Test Set Forecast',
              save_path='../outputs/figures/xgb_forecast.png')

In [ ]:
# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(20)
fi.plot(kind='barh', figsize=(10, 7), color='steelblue', edgecolor='white')
plt.gca().invert_yaxis()
plt.title('XGBoost — Top 20 Feature Importances')
plt.tight_layout()
plt.savefig('../outputs/figures/xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()